# 02c: Residual Stream: Circuit Analysis

## Overview
Analyzes residual stream representations under **mean ablation** (circuit mode).
Non-circuit edges have their outputs replaced with dataset-mean activations,
isolating the computational path through the ACDC-discovered circuit.

## Key Questions
1. How does ablation change residual stream geometry?
2. Does the circuit preserve band separability?
3. Which layers/bands lose the most representational structure?

## Sections
1. Circuit residual geometry (norms, participation ratio)
2. Circuit separation ratio trajectory
3. Circuit probe trajectory (linear probe on ablated residual stream)
4. Base vs Circuit comparison (CKA, delta norms, probe overlay)
5. Cross-model circuit comparison
6. Cross-band variation in circuit representation preservation

## Data Sources
- Base activations: `outputs/extraction/activations/`
- Circuit activations: `outputs/extraction/circuit_activations/`
- Base residual analysis: `outputs/residual_stream/base/analysis/`

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from functools import partial as _partial
from sklearn.decomposition import PCA

sys.path.insert(0, str(Path.cwd()))

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_RANK,
    MODEL_INFO,
    BAND_COLORS,
    BAND_NAMES,
    MODEL_COLORS,
    MODEL_CAPACITY,
    MODEL_D_MODEL,
    ACTIVATIONS_DIR,
    RANDOM_SEED,
    N_PERMUTATIONS,
    CV_FOLDS,
    get_domain_dirs,
)
from utils.data_loading import (
    load_extracted_activations,
    save_analysis,
    build_representational_df,
    load_domain_csv,
)
from utils.circuit_loading import (
    load_circuit_activations,
    load_base_and_circuit,
    load_prune_scores,
    get_circuit_mask,
    get_circuit_summary,
)
from utils.geometry import (
    compute_band_centroids,
    compute_centroid_distances,
    compute_within_band_spread,
    compute_separation_ratio,
    compute_participation_ratio,
    compute_isotropy,
    linear_cka,
)
from utils.probing import train_probe
from utils.plotting import (
    setup_plotting,
    save_figure,
    plot_probe_trajectory,
    plot_scaling_panel,
)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

setup_plotting()

# Circuit analysis dirs
CIRCUIT_ANALYSIS, CIRCUIT_VIZ = get_domain_dirs("residual_stream", "circuit")
COMP_ANALYSIS, COMP_VIZ = get_domain_dirs("residual_stream", "comparison")
save_analysis_circuit = _partial(save_analysis, analysis_dir=CIRCUIT_ANALYSIS)
save_figure_circuit = _partial(save_figure, viz_dir=CIRCUIT_VIZ)
save_analysis_comp = _partial(save_analysis, analysis_dir=COMP_ANALYSIS)
save_figure_comp = _partial(save_figure, viz_dir=COMP_VIZ)

PROBE_PCA_DIM = 50  # PCA dimensionality reduction before probing (speed optimization)

print(f"Models: {MODELS}")
print(f"Circuit analysis: {CIRCUIT_ANALYSIS}")
print(f"Comparison analysis: {COMP_ANALYSIS}")

Models: ['pythia-70m', 'pythia-160m', 'pythia-410m', 'pythia-1b', 'pythia-1.4b']
Circuit analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/residual_stream/circuit/analysis
Comparison analysis: LSC_circuit_analysis/03_Phase_Representational/outputs/residual_stream/comparison/analysis


## 1. Circuit Residual Geometry

Compute norms, participation ratio, and isotropy of the residual stream
at prediction position under mean ablation at each layer.

In [2]:
rows_geom = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for band in BANDS:
        for draw in DRAWS:
            try:
                data = load_circuit_activations(model, band, draw)
            except FileNotFoundError:
                continue
            resid = data["resid_post_predpos"]  # (N, n_layers, d_model)

            for layer in range(n_layers):
                X = resid[:, layer, :]  # (N, d_model)
                norms = np.linalg.norm(X, axis=1)
                pr = compute_participation_ratio(X)
                iso = compute_isotropy(X)

                rows_geom.append(
                    {
                        "model": model,
                        "band": band,
                        "draw": draw,
                        "layer": layer,
                        "mean_norm": float(norms.mean()),
                        "std_norm": float(norms.std()),
                        "participation_ratio": pr,
                        "isotropy": iso,
                    }
                )

df_circuit_geom = pd.DataFrame(rows_geom)
save_analysis_circuit(df_circuit_geom, "02c_circuit_resid_geometry.csv")
print(f"Circuit geometry: {len(df_circuit_geom)} rows")
df_circuit_geom.head()

Circuit geometry: 1230 rows


,model,band,draw,layer,mean_norm,std_norm,participation_ratio,isotropy
0,pythia-70m,low,draw_1,0,10.438556,0.924033,57.345038,0.425156
1,pythia-70m,low,draw_1,1,11.904775,1.366828,28.384987,0.384228
2,pythia-70m,low,draw_1,2,20.329344,15.371581,1.471335,0.409382
3,pythia-70m,low,draw_1,3,20.801966,14.752020,1.519669,0.426539
4,pythia-70m,low,draw_1,4,16.674503,4.752521,5.254371,0.358166


In [3]:
# Visualization: geometry trajectory per model
for model in MODELS:
    df_m = df_circuit_geom[df_circuit_geom["model"] == model]
    if df_m.empty:
        continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for band in BANDS:
        df_b = (
            df_m[df_m["band"] == band]
            .groupby("layer")
            .mean(numeric_only=True)
            .reset_index()
        )
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)
        axes[0].plot(
            df_b["layer"], df_b["mean_norm"], color=color, label=label, marker="o", ms=3
        )
        axes[1].plot(
            df_b["layer"],
            df_b["participation_ratio"],
            color=color,
            label=label,
            marker="o",
            ms=3,
        )
        axes[2].plot(
            df_b["layer"], df_b["isotropy"], color=color, label=label, marker="o", ms=3
        )

    axes[0].set_title("Mean Norm (Circuit)")
    axes[1].set_title("Participation Ratio (Circuit)")
    axes[2].set_title("Isotropy (Circuit)")
    for ax in axes:
        ax.set_xlabel("Layer")
        ax.legend(fontsize=8)

    fig.suptitle(f"Circuit Residual Geometry: {model}", y=1.02)
    fig.tight_layout()
    save_figure_circuit(fig, f"viz_02c_01_circuit_geometry_{model}.png")

## 2. Circuit Separation Ratio Trajectory

Band separability in circuit-constrained representations at each layer.
Separation ratio = mean inter-band centroid distance / mean within-band spread.

In [4]:
rows_sep = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for draw in DRAWS:
        # Collect all bands for this model/draw
        band_data = {}
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
                band_data[band] = data["resid_post_predpos"]
            except FileNotFoundError:
                pass

        if len(band_data) < 2:
            continue

        for layer in range(n_layers):
            # Stack all bands' activations at this layer
            all_X = []
            all_labels = []
            for band, resid in band_data.items():
                X = resid[:, layer, :]
                all_X.append(X)
                all_labels.extend([band] * len(X))

            all_X = np.vstack(all_X)
            all_labels = np.array(all_labels)

            centroids = compute_band_centroids(all_X, all_labels)
            centroid_dists = compute_centroid_distances(centroids)
            spreads = compute_within_band_spread(all_X, all_labels)
            sep_ratio = compute_separation_ratio(centroid_dists, spreads)

            # centroid_dists is a DataFrame; extract off-diagonal mean
            mask = np.ones(centroid_dists.shape, dtype=bool)
            np.fill_diagonal(mask, False)
            mean_cd = (
                float(centroid_dists.values[mask].mean())
                if centroid_dists.size > 0
                else 0
            )
            mean_sp = float(np.mean(list(spreads.values()))) if spreads else 0

            rows_sep.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "separation_ratio": sep_ratio,
                    "mean_centroid_dist": mean_cd,
                    "mean_spread": mean_sp,
                }
            )

df_circuit_sep = pd.DataFrame(rows_sep)
save_analysis_circuit(df_circuit_sep, "02c_circuit_separation_trajectory.csv")
print(f"Circuit separation: {len(df_circuit_sep)} rows")

Circuit separation: 246 rows


In [5]:
# Visualization: separation ratio trajectory
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    df_m = df_circuit_sep[df_circuit_sep["model"] == model]
    if df_m.empty:
        continue
    df_mean = df_m.groupby("layer")["separation_ratio"].mean().reset_index()
    color = MODEL_COLORS.get(model, "gray")
    ax.plot(
        df_mean["layer"],
        df_mean["separation_ratio"],
        color=color,
        label=model,
        marker="o",
        ms=4,
    )

ax.set_xlabel("Layer")
ax.set_ylabel("Separation Ratio")
ax.set_title("Circuit Separation Ratio Trajectory")
ax.legend()
fig.tight_layout()
save_figure_circuit(fig, "viz_02c_02_circuit_separation_trajectory.png")

## 3. Circuit Probe Trajectory

Linear probe accuracy on circuit-mode residual stream at each layer.
Can the circuit still classify frequency band from its residual stream?

In [6]:
rows_probe = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    d_model = MODEL_D_MODEL[model]
    n_pca = min(PROBE_PCA_DIM, d_model)
    print(f"\n{model} ({n_layers} layers, d_model={d_model}, PCA->{n_pca}):")

    for draw in DRAWS:
        # Collect all bands
        band_data = {}
        for band in BANDS:
            try:
                data = load_circuit_activations(model, band, draw)
                band_data[band] = data["resid_post_predpos"]
            except FileNotFoundError:
                pass

        if len(band_data) < 2:
            continue

        for layer in range(n_layers):
            all_X = []
            all_labels = []
            for band, resid in band_data.items():
                all_X.append(resid[:, layer, :])
                all_labels.extend([band] * resid.shape[0])

            all_X = np.vstack(all_X)
            all_labels = np.array(all_labels)

            # PCA dimensionality reduction for speed
            if all_X.shape[1] > n_pca:
                pca = PCA(n_components=n_pca, random_state=RANDOM_SEED)
                all_X = pca.fit_transform(all_X)

            result = train_probe(all_X, all_labels, n_folds=CV_FOLDS)
            rows_probe.append(
                {
                    "model": model,
                    "draw": draw,
                    "layer": layer,
                    "accuracy": result["accuracy"],
                    "std": result["std"],
                }
            )

        # Report peak for this draw
        draw_rows = [r for r in rows_probe if r["model"] == model and r["draw"] == draw]
        if draw_rows:
            peak = max(draw_rows, key=lambda r: r["accuracy"])
            print(f"  {draw}: peak = {peak['accuracy']:.3f} at layer {peak['layer']}")

df_circuit_probe = pd.DataFrame(rows_probe)
save_analysis_circuit(df_circuit_probe, "02c_circuit_probe_trajectory.csv")
print(f"\nCircuit probe: {len(df_circuit_probe)} rows")


pythia-70m (6 layers, d_model=512, PCA->50):


  draw_1: peak = 0.620 at layer 3


  draw_2: peak = 0.629 at layer 4


  draw_3: peak = 0.589 at layer 3

pythia-160m (12 layers, d_model=768, PCA->50):


  draw_1: peak = 0.697 at layer 6


...

  draw_2: peak = 0.743 at layer 13


  draw_3: peak = 0.707 at layer 13

pythia-1.4b (24 layers, d_model=2048, PCA->50):


  draw_1: peak = 0.724 at layer 21


  draw_2: peak = 0.780 at layer 22


  draw_3: peak = 0.746 at layer 20

Circuit probe: 246 rows


In [7]:
# Visualization: circuit probe trajectory
fig, ax = plt.subplots(figsize=(12, 6))
for model in MODELS:
    df_m = df_circuit_probe[df_circuit_probe["model"] == model]
    if df_m.empty:
        continue
    df_mean = (
        df_m.groupby("layer").agg({"accuracy": "mean", "std": "mean"}).reset_index()
    )
    color = MODEL_COLORS.get(model, "gray")
    ax.plot(
        df_mean["layer"],
        df_mean["accuracy"],
        color=color,
        label=model,
        marker="o",
        ms=4,
    )
    ax.fill_between(
        df_mean["layer"],
        df_mean["accuracy"] - df_mean["std"],
        df_mean["accuracy"] + df_mean["std"],
        color=color,
        alpha=0.15,
    )

ax.axhline(y=0.2, color="gray", linestyle="--", alpha=0.5, label="Chance (1/5)")
ax.set_xlabel("Layer")
ax.set_ylabel("Probe Accuracy")
ax.set_title("Circuit Probe Trajectory")
ax.legend()
ax.set_ylim(0, 1)
fig.tight_layout()
save_figure_circuit(fig, "viz_02c_03_circuit_probe_trajectory.png")

## 4. Base vs Circuit Comparison

Compare base model and circuit representations:
- Probe accuracy overlay
- CKA between base and circuit residual streams
- Delta norms (||circuit - base|| per layer)
- Separation ratio comparison

In [8]:
# Load base probe trajectory for comparison
try:
    df_base_probe = load_domain_csv(
        "residual_stream", "base", "02_probe_trajectory.csv"
    )
    print(f"Base probe trajectory: {len(df_base_probe)} rows")
except FileNotFoundError:
    print("Base probe trajectory not found, skipping comparison")
    df_base_probe = pd.DataFrame()

Base probe trajectory: 246 rows


In [9]:
# Probe trajectory overlay: base vs circuit
if not df_base_probe.empty and not df_circuit_probe.empty:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(12, 6))

        # Base (solid)
        df_base_m = df_base_probe[df_base_probe["model"] == model]
        if not df_base_m.empty:
            df_mean = df_base_m.groupby("layer")["accuracy"].mean().reset_index()
            ax.plot(
                df_mean["layer"],
                df_mean["accuracy"],
                color="steelblue",
                label="Base",
                linewidth=2,
                marker="o",
                ms=4,
            )

        # Circuit (dashed)
        df_circ_m = df_circuit_probe[df_circuit_probe["model"] == model]
        if not df_circ_m.empty:
            df_mean = df_circ_m.groupby("layer")["accuracy"].mean().reset_index()
            ax.plot(
                df_mean["layer"],
                df_mean["accuracy"],
                color="coral",
                label="Circuit",
                linewidth=2,
                linestyle="--",
                marker="s",
                ms=4,
            )

        ax.axhline(y=0.2, color="gray", linestyle=":", alpha=0.5)
        ax.set_xlabel("Layer")
        ax.set_ylabel("Probe Accuracy")
        ax.set_title(f"Base vs Circuit Probe: {model}")
        ax.legend()
        ax.set_ylim(0, 1)
        fig.tight_layout()
        save_figure_comp(fig, f"viz_02c_04_probe_comparison_{model}.png")

In [10]:
# CKA between base and circuit residual streams per layer
rows_cka = []

for model in MODELS:
    n_layers = MODEL_INFO[model]["n_layers"]
    for band in BANDS:
        for draw in DRAWS:
            try:
                base, circuit = load_base_and_circuit(model, band, draw)
            except FileNotFoundError:
                continue

            base_resid = base["resid_post_predpos"]
            circ_resid = circuit["resid_post_predpos"]

            for layer in range(n_layers):
                X_base = base_resid[:, layer, :]
                X_circ = circ_resid[:, layer, :]

                cka = linear_cka(X_base, X_circ)
                delta_norm = float(np.mean(np.linalg.norm(X_circ - X_base, axis=1)))

                rows_cka.append(
                    {
                        "model": model,
                        "band": band,
                        "draw": draw,
                        "layer": layer,
                        "cka_base_circuit": cka,
                        "mean_delta_norm": delta_norm,
                    }
                )

df_cka_comp = pd.DataFrame(rows_cka)
save_analysis_comp(df_cka_comp, "02c_base_circuit_cka.csv")
print(f"CKA comparison: {len(df_cka_comp)} rows")

CKA comparison: 1230 rows


In [11]:
# Visualization: CKA and delta norm per model
for model in MODELS:
    df_m = df_cka_comp[df_cka_comp["model"] == model]
    if df_m.empty:
        continue

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for band in BANDS:
        df_b = (
            df_m[df_m["band"] == band]
            .groupby("layer")
            .mean(numeric_only=True)
            .reset_index()
        )
        color = BAND_COLORS.get(band, "gray")
        label = BAND_NAMES.get(band, band)
        ax1.plot(
            df_b["layer"],
            df_b["cka_base_circuit"],
            color=color,
            label=label,
            marker="o",
            ms=3,
        )
        ax2.plot(
            df_b["layer"],
            df_b["mean_delta_norm"],
            color=color,
            label=label,
            marker="o",
            ms=3,
        )

    ax1.set_ylabel("CKA(Base, Circuit)")
    ax1.set_title("Base-Circuit CKA")
    ax1.set_ylim(0, 1.05)
    ax2.set_ylabel("Mean ||Circuit - Base||")
    ax2.set_title("Activation Delta Norm")
    for ax in [ax1, ax2]:
        ax.set_xlabel("Layer")
        ax.legend(fontsize=8)

    fig.suptitle(f"Base vs Circuit Comparison: {model}", y=1.02)
    fig.tight_layout()
    save_figure_comp(fig, f"viz_02c_05_cka_delta_norm_{model}.png")

In [12]:
# Separation ratio comparison: base vs circuit
try:
    df_base_sep = load_domain_csv(
        "residual_stream", "base", "02_separation_trajectory.csv"
    )
except FileNotFoundError:
    df_base_sep = pd.DataFrame()

if not df_base_sep.empty and not df_circuit_sep.empty:
    for model in MODELS:
        fig, ax = plt.subplots(figsize=(10, 6))

        df_base_m = df_base_sep[df_base_sep["model"] == model]
        if not df_base_m.empty:
            df_mean = (
                df_base_m.groupby("layer")["separation_ratio"].mean().reset_index()
            )
            ax.plot(
                df_mean["layer"],
                df_mean["separation_ratio"],
                color="steelblue",
                label="Base",
                linewidth=2,
                marker="o",
                ms=4,
            )

        df_circ_m = df_circuit_sep[df_circuit_sep["model"] == model]
        if not df_circ_m.empty:
            df_mean = (
                df_circ_m.groupby("layer")["separation_ratio"].mean().reset_index()
            )
            ax.plot(
                df_mean["layer"],
                df_mean["separation_ratio"],
                color="coral",
                label="Circuit",
                linewidth=2,
                linestyle="--",
                marker="s",
                ms=4,
            )

        ax.set_xlabel("Layer")
        ax.set_ylabel("Separation Ratio")
        ax.set_title(f"Separation Ratio: Base vs Circuit: {model}")
        ax.legend()
        fig.tight_layout()
        save_figure_comp(fig, f"viz_02c_06_separation_comparison_{model}.png")

## 5. Cross-Model Circuit Comparison

In [13]:
# Summary statistics per model
rows_summary = []

for model in MODELS:
    # Circuit probe
    df_m = df_circuit_probe[df_circuit_probe["model"] == model]
    if df_m.empty:
        continue
    peak_acc = df_m.groupby("layer")["accuracy"].mean().max()
    peak_layer = df_m.groupby("layer")["accuracy"].mean().idxmax()

    # Circuit separation
    df_s = df_circuit_sep[df_circuit_sep["model"] == model]
    max_sep = (
        df_s.groupby("layer")["separation_ratio"].mean().max()
        if not df_s.empty
        else None
    )

    # CKA
    df_c = df_cka_comp[df_cka_comp["model"] == model]
    mean_cka = df_c["cka_base_circuit"].mean() if not df_c.empty else None

    rows_summary.append(
        {
            "model": model,
            "model_capacity": MODEL_CAPACITY.get(model, 0),
            "circuit_peak_probe_acc": peak_acc,
            "circuit_peak_probe_layer": peak_layer,
            "circuit_max_separation": max_sep,
            "mean_cka_base_circuit": mean_cka,
        }
    )

df_summary = pd.DataFrame(rows_summary)
save_analysis_circuit(df_summary, "02c_circuit_summary.csv")
print(df_summary.to_string(index=False))

      model  model_capacity  circuit_peak_probe_acc  circuit_peak_probe_layer  circuit_max_separation  mean_cka_base_circuit
 pythia-70m              70                0.611259                         3               12.105699               0.997731
pythia-160m             160                0.689481                         6               12.385745               0.986592
pythia-410m             410                0.678815                        21               13.969808               0.972895
  pythia-1b            1000                0.731852                        13               17.467688               0.959402
pythia-1.4b            1400                0.744000                        21               20.319380               0.902683


In [14]:
# Cross-model scaling: circuit metrics vs model size
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

if not df_summary.empty:
    x = df_summary["model_capacity"]
    axes[0].plot(x, df_summary["circuit_peak_probe_acc"], "o-", color="coral")
    axes[0].set_title("Circuit Peak Probe Accuracy")
    axes[0].set_ylabel("Accuracy")

    axes[1].plot(x, df_summary["circuit_max_separation"], "o-", color="coral")
    axes[1].set_title("Circuit Max Separation Ratio")
    axes[1].set_ylabel("Separation Ratio")

    axes[2].plot(x, df_summary["mean_cka_base_circuit"], "o-", color="coral")
    axes[2].set_title("Mean CKA(Base, Circuit)")
    axes[2].set_ylabel("CKA")
    axes[2].set_ylim(0, 1.05)

    for ax in axes:
        ax.set_xlabel("Model Capacity (M)")
        ax.set_xscale("log")

fig.suptitle("Circuit Representational Metrics vs Model Size", y=1.02)
fig.tight_layout()
save_figure_circuit(fig, "viz_02c_07_cross_model_scaling.png")

## 6. Cross-Band Variation

Do some bands' circuits preserve representations better than others?

In [15]:
# Per-band CKA (averaged over layers and draws)
if not df_cka_comp.empty:
    df_band_cka = (
        df_cka_comp.groupby(["model", "band"])
        .agg(
            mean_cka=("cka_base_circuit", "mean"),
            mean_delta_norm=("mean_delta_norm", "mean"),
        )
        .reset_index()
    )

    save_analysis_comp(df_band_cka, "02c_per_band_preservation.csv")

    # Heatmap: model x band CKA
    pivot = df_band_cka.pivot(index="model", columns="band", values="mean_cka")
    pivot = pivot.reindex(index=MODELS, columns=BANDS)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        square=True,
        linewidths=0,
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_title("Mean CKA(Base, Circuit) by Model x Band")
    fig.tight_layout()
    save_figure_comp(fig, "viz_02c_08_band_preservation_heatmap.png")

In [16]:
# Master summary
rows_master = []
for model in MODELS:
    for band in BANDS:
        for draw in DRAWS:
            # Circuit summary
            try:
                ps = load_prune_scores(model, band, draw)
                cs = get_circuit_summary(ps)
            except FileNotFoundError:
                continue

            # CKA
            cka_rows = df_cka_comp[
                (df_cka_comp["model"] == model)
                & (df_cka_comp["band"] == band)
                & (df_cka_comp["draw"] == draw)
            ]
            mean_cka = (
                cka_rows["cka_base_circuit"].mean() if not cka_rows.empty else None
            )

            rows_master.append(
                {
                    "model": model,
                    "band": band,
                    "draw": draw,
                    "circuit_edges": cs["total_edges"],
                    "mean_cka_base_circuit": mean_cka,
                    "frequency_rank": FREQUENCY_RANK.get(band),
                    "model_capacity": MODEL_CAPACITY.get(model),
                }
            )

df_master = pd.DataFrame(rows_master)
save_analysis_circuit(df_master, "02c_master_circuit_residual.csv")
print(f"Master: {len(df_master)} rows")
df_master.head()

Master: 75 rows


,model,band,draw,circuit_edges,mean_cka_base_circuit,frequency_rank,model_capacity
0,pythia-70m,low,draw_1,380,0.996752,1.0,70
1,pythia-70m,low,draw_2,396,0.997327,1.0,70
2,pythia-70m,low,draw_3,391,0.996618,1.0,70
3,pythia-70m,medium,draw_1,395,0.997515,2.0,70
4,pythia-70m,medium,draw_2,392,0.999811,2.0,70
